# Haydock Cross-Reference Analysis

This notebook explores the Haydock Catholic Bible Commentary cross-references added in Phase 2, with focus on:

1. **Source Comparison**: How TSK and Haydock cross-references compare
2. **Canon Analysis**: What cross-references are gained/lost between the Protestant 66-book canon and the Catholic 73-book canon
3. **Visualizations**: Interactive graphs of Haydock connections and deuterocanonical integration

## Setup & Connection

In [1]:
import sys
sys.path.insert(0, '../..')

from src.db.connection import get_connection
from pyvis.network import Network

In [2]:
conn = get_connection()
conn.connect()
conn.verify()
print("Connected to Neo4j!")

Connected to Neo4j!


In [3]:
# Deuterocanonical books (Catholic canon additions)
DEUTEROCANONICAL_BOOKS = ['TOB', 'JDT', 'WIS', 'SIR', 'BAR', '1MA', '2MA']

# Full names for display
DC_BOOK_NAMES = {
    'TOB': 'Tobit',
    'JDT': 'Judith', 
    'WIS': 'Wisdom',
    'SIR': 'Sirach',
    'BAR': 'Baruch',
    '1MA': '1 Maccabees',
    '2MA': '2 Maccabees'
}

In [4]:
# Color scheme by book category (same as 01_exploration)
BOOK_COLORS = {
    # Pentateuch (Torah) - Blue
    "GEN": "#3498db", "EXO": "#3498db", "LEV": "#3498db", "NUM": "#3498db", "DEU": "#3498db",
    # Historical Books - Green
    "JOS": "#27ae60", "JDG": "#27ae60", "RUT": "#27ae60", "1SA": "#27ae60", "2SA": "#27ae60",
    "1KI": "#27ae60", "2KI": "#27ae60", "1CH": "#27ae60", "2CH": "#27ae60", "EZR": "#27ae60",
    "NEH": "#27ae60", "EST": "#27ae60",
    # Deuterocanonical - Teal
    "TOB": "#16a085", "JDT": "#16a085", "1MA": "#16a085", "2MA": "#16a085",
    "WIS": "#16a085", "SIR": "#16a085", "BAR": "#16a085",
    # Wisdom/Poetry - Gold
    "JOB": "#f39c12", "PSA": "#f39c12", "PRO": "#f39c12", "ECC": "#f39c12", "SNG": "#f39c12",
    # Major Prophets - Red
    "ISA": "#e74c3c", "JER": "#e74c3c", "LAM": "#e74c3c", "EZK": "#e74c3c", "DAN": "#e74c3c",
    # Minor Prophets - Orange
    "HOS": "#e67e22", "JOL": "#e67e22", "AMO": "#e67e22", "OBA": "#e67e22", "JON": "#e67e22",
    "MIC": "#e67e22", "NAM": "#e67e22", "HAB": "#e67e22", "ZEP": "#e67e22", "HAG": "#e67e22",
    "ZEC": "#e67e22", "MAL": "#e67e22",
    # Gospels - Purple
    "MAT": "#9b59b6", "MRK": "#9b59b6", "LUK": "#9b59b6", "JHN": "#9b59b6",
    # Acts - Light Purple
    "ACT": "#8e44ad",
    # Pauline Epistles - Pink
    "ROM": "#e91e63", "1CO": "#e91e63", "2CO": "#e91e63", "GAL": "#e91e63", "EPH": "#e91e63",
    "PHP": "#e91e63", "COL": "#e91e63", "1TH": "#e91e63", "2TH": "#e91e63", "1TI": "#e91e63",
    "2TI": "#e91e63", "TIT": "#e91e63", "PHM": "#e91e63",
    # General Epistles - Cyan
    "HEB": "#00bcd4", "JAS": "#00bcd4", "1PE": "#00bcd4", "2PE": "#00bcd4",
    "1JN": "#00bcd4", "2JN": "#00bcd4", "3JN": "#00bcd4", "JUD": "#00bcd4",
    # Revelation - Dark Red
    "REV": "#c0392b",
}

def get_book_color(book_id):
    """Get color for a book, with fallback."""
    return BOOK_COLORS.get(book_id, "#95a5a6")

## Source Distribution Overview

First, let's understand the overall distribution of cross-references by source.

In [5]:
with conn.session() as session:
    result = session.run("""
        MATCH ()-[r:CROSS_REFERENCES]->()
        RETURN
          CASE
            WHEN 'TSK' IN r.sources AND 'Haydock' IN r.sources THEN 'Both'
            WHEN 'TSK' IN r.sources THEN 'TSK-only'
            WHEN 'Haydock' IN r.sources THEN 'Haydock-only'
            ELSE 'Unknown'
          END AS source_type,
          count(r) AS count
        ORDER BY count DESC
    """)
    
    source_counts = {}
    total = 0
    print("Cross-Reference Distribution by Source:")
    print("=" * 45)
    for record in result:
        source_counts[record['source_type']] = record['count']
        total += record['count']
    
    for source_type, count in source_counts.items():
        pct = (count / total) * 100
        print(f"  {source_type:15} {count:>10,} ({pct:5.1f}%)")
    print("=" * 45)
    print(f"  {'TOTAL':15} {total:>10,}")

Cross-Reference Distribution by Source:
  TSK-only           572,152 ( 98.9%)
  Both                 4,105 (  0.7%)
  Haydock-only         2,477 (  0.4%)
  TOTAL              578,734


### What This Tells Us

- **TSK-only**: Edges from Treasury of Scripture Knowledge, based largely on English word matching
- **Haydock-only**: Edges unique to Haydock's Catholic commentary, often reflecting patristic/theological connections
- **Both**: Edges attested by both sources, representing high-confidence connections

## TSK vs Haydock Comparison

### High-Confidence Connections (Both Sources Agree)

Edges attested by both TSK and Haydock represent particularly strong connections.

In [6]:
with conn.session() as session:
    result = session.run("""
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'TSK' IN r.sources AND 'Haydock' IN r.sources
        RETURN a.id AS from_id, b.id AS to_id, r.votes AS votes,
               a.text AS from_text, b.text AS to_text
        ORDER BY r.votes DESC
        LIMIT 10
    """)
    
    print("Top 10 Connections Attested by Both TSK and Haydock:")
    print("(Sorted by TSK votes)")
    print("=" * 80)
    for record in result:
        print(f"\n{record['from_id']} -> {record['to_id']} (votes: {record['votes']})")
        print(f"  FROM: {record['from_text'][:70]}...")
        print(f"  TO:   {record['to_text'][:70]}...")

Top 10 Connections Attested by Both TSK and Haydock:
(Sorted by TSK votes)

JHN-3-16 -> 1JN-4-9 (votes: 618)
  FROM: For God so loved the world that he gave his only-begotten Son, so that...
  TO:   The love of God was made apparent to us in this way: that God sent his...

1SA-15-22 -> HOS-6-6 (votes: 408)
  FROM: And Samuel said: “Does the Lord want holocausts and victims, and not i...
  TO:   For I desired mercy and not sacrifice, and knowledge of God more than ...

MAT-4-4 -> DEU-8-3 (votes: 349)
  FROM: And in response he said, “It has been written: ‘Not by bread alone sha...
  TO:   He afflicted you with need, and he gave you Manna as your food, which ...

LUK-11-9 -> MRK-11-24 (votes: 323)
  FROM: And so I say to you: Ask, and it shall be given to you. Seek, and you ...
  TO:   For this reason, I say to you, all things whatsoever that you ask for ...

1CO-2-9 -> ISA-64-4 (votes: 298)
  FROM: But this is just as it has been written: “The eye has not seen, and th...
  TO:   From ag

### Top Verses by Haydock-Only References

Which verses are most referenced by Haydock connections that TSK doesn't have?

In [7]:
with conn.session() as session:
    result = session.run("""
        MATCH (v:Verse)<-[r:CROSS_REFERENCES]-()
        WHERE 'Haydock' IN r.sources AND NOT 'TSK' IN r.sources
        WITH v, count(r) AS refs
        ORDER BY refs DESC
        LIMIT 15
        RETURN v.id AS verse_id, v.book_name AS book, refs, v.text AS text
    """)
    
    print("Top 15 Verses by Haydock-Only Incoming References:")
    print("=" * 80)
    for record in result:
        print(f"\n{record['verse_id']} ({record['refs']} refs)")
        print(f"  {record['text'][:75]}...")

Top 15 Verses by Haydock-Only Incoming References:

EXO-14-22 (8 refs)
  And the sons of Israel went in through the midst of the dried sea. For the ...

WIS-6-8 (8 refs)
  For the Lord will not exempt anyone’s character, nor will he stand in awe o...

TOB-1-21 (7 refs)
  And then, when king Sennacherib had returned from Judea, fleeing the scourg...

2MA-8-19 (6 refs)
  Moreover, he reminded them also of the assistance of God which their parent...

SIR-35-15 (6 refs)
  And do not be willing to consider an unjust sacrifice. For the Lord is the ...

PSA-117-22 (6 refs)
  The stone which the builders have rejected, this has become the head of the...

PSA-109-1 (5 refs)
  A Psalm of David.  The Lord said to my Lord, “Sit at my right hand, until I...

PSA-109-4 (5 refs)
  The Lord has sworn, and he will not repent: “You are a priest forever, acco...

EXO-12-29 (5 refs)
  Then it happened, in the middle of the night: the Lord struck down every fi...

GEN-2-7 (5 refs)
  And then the Lord God f

### Top Verses by TSK-Only References

Which verses are most referenced by TSK connections that Haydock doesn't have?

In [8]:
with conn.session() as session:
    result = session.run("""
        MATCH (v:Verse)<-[r:CROSS_REFERENCES]-()
        WHERE 'TSK' IN r.sources AND NOT 'Haydock' IN r.sources
        WITH v, count(r) AS refs
        ORDER BY refs DESC
        LIMIT 15
        RETURN v.id AS verse_id, v.book_name AS book, refs, v.text AS text
    """)
    
    print("Top 15 Verses by TSK-Only Incoming References:")
    print("=" * 80)
    for record in result:
        print(f"\n{record['verse_id']} ({record['refs']} refs)")
        print(f"  {record['text'][:75]}...")

Top 15 Verses by TSK-Only Incoming References:

ISA-9-7 (183 refs)
  His reign will be increased, and there will be no end to his peace. He will...

TIT-2-14 (177 refs)
  He gave himself for our sake, so that he might redeem us from all iniquity,...

REV-5-9 (171 refs)
  And they were singing a new canticle, saying: “O Lord, you are worthy to re...

1PE-2-9 (170 refs)
  But you are a chosen generation, a royal priesthood, a holy nation, an acqu...

ISA-9-6 (165 refs)
  For unto us a child is born, and unto us a son is given. And leadership is ...

MAT-28-20 (162 refs)
  teaching them to observe all that I have ever commanded you. And behold, I ...

REV-19-20 (159 refs)
  And the beast was apprehended, and with him the false prophetess, who in hi...

2CO-5-21 (147 refs)
  For God made him who did not know sin to be sin for us, so that we might be...

TIT-3-5 (146 refs)
  And he saved us, not by works of justice that we had done, but, in accord w...

ISA-55-7 (139 refs)
  Let the impious

### Book-Level Source Comparison

Which books does Haydock emphasize more than TSK, and vice versa?

In [9]:
with conn.session() as session:
    # Get Haydock-only refs by target book
    result = session.run("""
        MATCH ()-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'Haydock' IN r.sources AND NOT 'TSK' IN r.sources
        RETURN b.book_id AS book, count(r) AS haydock_only
        ORDER BY haydock_only DESC
    """)
    haydock_only = {r['book']: r['haydock_only'] for r in result}
    
    # Get TSK-only refs by target book  
    result = session.run("""
        MATCH ()-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'TSK' IN r.sources AND NOT 'Haydock' IN r.sources
        RETURN b.book_id AS book, count(r) AS tsk_only
        ORDER BY tsk_only DESC
    """)
    tsk_only = {r['book']: r['tsk_only'] for r in result}
    
    # Calculate ratio (Haydock / TSK) for each book
    all_books = set(haydock_only.keys()) | set(tsk_only.keys())
    ratios = []
    for book in all_books:
        h = haydock_only.get(book, 0)
        t = tsk_only.get(book, 0)
        if t > 0:
            ratio = h / t
        else:
            ratio = float('inf') if h > 0 else 0
        ratios.append((book, h, t, ratio))
    
    # Sort by ratio (highest = Haydock emphasis)
    ratios.sort(key=lambda x: x[3], reverse=True)
    
    print("Books Most Emphasized by Haydock (vs TSK):")
    print("(Ratio = Haydock-only refs / TSK-only refs)")
    print("=" * 55)
    print(f"{'Book':6} {'Haydock-only':>12} {'TSK-only':>10} {'Ratio':>10}")
    print("-" * 55)
    for book, h, t, ratio in ratios[:15]:
        dc_marker = " [DC]" if book in DEUTEROCANONICAL_BOOKS else ""
        ratio_str = f"{ratio:.2f}" if ratio != float('inf') else "inf"
        print(f"{book:6} {h:>12,} {t:>10,} {ratio_str:>10}{dc_marker}")

Books Most Emphasized by Haydock (vs TSK):
(Ratio = Haydock-only refs / TSK-only refs)
Book   Haydock-only   TSK-only      Ratio
-------------------------------------------------------
2MA              25          0        inf [DC]
TOB              30          0        inf [DC]
JDT              13          0        inf [DC]
WIS              70          0        inf [DC]
BAR              16          0        inf [DC]
SIR             167          0        inf [DC]
1MA              32          0        inf [DC]
EXO             154     18,938       0.01
MRK              68      8,367       0.01
HAG               5        669       0.01
GEN             142     19,104       0.01
EST              11      1,517       0.01
PSA             259     36,843       0.01
JUD               5        930       0.01
1PE              26      4,938       0.01


In [10]:
# Now show books TSK emphasizes more
ratios.sort(key=lambda x: x[3])  # Sort ascending (lowest = TSK emphasis)

print("\nBooks Most Emphasized by TSK (vs Haydock):")
print("=" * 55)
print(f"{'Book':6} {'Haydock-only':>12} {'TSK-only':>10} {'Ratio':>10}")
print("-" * 55)
for book, h, t, ratio in ratios[:15]:
    if t > 0:  # Only books with TSK refs
        dc_marker = " [DC]" if book in DEUTEROCANONICAL_BOOKS else ""
        print(f"{book:6} {h:>12,} {t:>10,} {ratio:>10.2f}{dc_marker}")


Books Most Emphasized by TSK (vs Haydock):
Book   Haydock-only   TSK-only      Ratio
-------------------------------------------------------
LAM               0      2,605       0.00
TIT               0      1,882       0.00
2JN               0        271       0.00
NEH               0      5,112       0.00
2TH               0      1,921       0.00
SNG               0      1,313       0.00
3JN               0        249       0.00
NAM               0        805       0.00
PHM               0        349       0.00
HAB               1      1,397       0.00
ZEP               1      1,369       0.00
REV              15     15,891       0.00
MIC               3      3,048       0.00
ZEC               5      4,934       0.00
PHP               4      3,424       0.00


## Canon Comparison Analysis

The key question: What cross-references are gained when including the 7 deuterocanonical books?

**Protestant Canon**: 66 books (excludes TOB, JDT, WIS, SIR, BAR, 1MA, 2MA)  
**Catholic Canon**: 73 books (includes all deuterocanonicals)

### Cross-References Involving Deuterocanonical Books

These are connections that would be completely unavailable in a Protestant-canon-only analysis.

In [11]:
with conn.session() as session:
    # Total edges involving DC books (from either source)
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE a.book_id IN dc_books OR b.book_id IN dc_books
        RETURN count(r) AS total_dc_edges
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    total_dc = result.single()['total_dc_edges']
    
    # Edges where BOTH endpoints are in DC books
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE a.book_id IN dc_books AND b.book_id IN dc_books
        RETURN count(r) AS internal_dc_edges
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    internal_dc = result.single()['internal_dc_edges']
    
    # Edges connecting DC to non-DC (bridging edges)
    bridging = total_dc - internal_dc
    
    # Get total edges for context
    result = session.run("MATCH ()-[r:CROSS_REFERENCES]->() RETURN count(r) AS total")
    total_all = result.single()['total']
    
    print("Cross-References Involving Deuterocanonical Books:")
    print("=" * 55)
    print(f"  Total edges in database:            {total_all:>10,}")
    print(f"  Edges involving DC books:           {total_dc:>10,} ({total_dc/total_all*100:.1f}%)")
    print(f"    - Between DC books only:          {internal_dc:>10,}")
    print(f"    - Bridging DC to non-DC:          {bridging:>10,}")
    print("\n  => These edges are LOST without the deuterocanonical books")

Cross-References Involving Deuterocanonical Books:
  Total edges in database:               578,734
  Edges involving DC books:                  857 (0.1%)
    - Between DC books only:                 103
    - Bridging DC to non-DC:                 754

  => These edges are LOST without the deuterocanonical books


### Breakdown by Deuterocanonical Book

Which deuterocanonical books contribute the most cross-references?

In [12]:
with conn.session() as session:
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE a.book_id IN dc_books OR b.book_id IN dc_books
        WITH 
            CASE 
                WHEN a.book_id IN dc_books AND b.book_id IN dc_books THEN [a.book_id, b.book_id]
                WHEN a.book_id IN dc_books THEN [a.book_id]
                ELSE [b.book_id]
            END AS dc_book_list,
            r
        UNWIND dc_book_list AS dc_book
        RETURN dc_book, count(DISTINCT r) AS refs
        ORDER BY refs DESC
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    
    print("Cross-References by Deuterocanonical Book:")
    print("=" * 45)
    print(f"{'Book':6} {'Name':15} {'Refs':>10}")
    print("-" * 45)
    for record in result:
        book = record['dc_book']
        name = DC_BOOK_NAMES.get(book, book)
        print(f"{book:6} {name:15} {record['refs']:>10,}")

Cross-References by Deuterocanonical Book:
Book   Name                  Refs
---------------------------------------------
SIR    Sirach                 446
WIS    Wisdom                 214
TOB    Tobit                   70
1MA    1 Maccabees             61
2MA    2 Maccabees             48
BAR    Baruch                  30
JDT    Judith                  29


### Which Protestant-Canon Books Connect Most to Deuterocanonicals?

Which books from the shared canon have the strongest connections to the deuterocanonical books?

In [13]:
with conn.session() as session:
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE (a.book_id IN dc_books AND NOT b.book_id IN dc_books)
           OR (b.book_id IN dc_books AND NOT a.book_id IN dc_books)
        WITH 
            CASE 
                WHEN a.book_id IN dc_books THEN b.book_id
                ELSE a.book_id
            END AS non_dc_book,
            r
        RETURN non_dc_book AS book, count(r) AS connections
        ORDER BY connections DESC
        LIMIT 20
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    
    print("Protestant-Canon Books with Most DC Connections:")
    print("=" * 35)
    print(f"{'Book':6} {'Connections':>15}")
    print("-" * 35)
    for record in result:
        print(f"{record['book']:6} {record['connections']:>15,}")

Protestant-Canon Books with Most DC Connections:
Book       Connections
-----------------------------------
GEN                 88
DEU                 67
EXO                 64
PRO                 52
ISA                 38
PSA                 38
NUM                 37
2KI                 30
1SA                 26
LEV                 25
ROM                 23
MAT                 23
1KI                 23
JER                 16
JHN                 15
JOB                 14
DAN                 14
2CH                 14
HEB                 13
LUK                 13


### Haydock's Unique Contribution to Deuterocanonical Connections

Since TSK doesn't cover deuterocanonical books, all DC cross-references come from Haydock.

In [14]:
with conn.session() as session:
    # Verify: DC edges by source
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE a.book_id IN dc_books OR b.book_id IN dc_books
        RETURN
          CASE
            WHEN 'TSK' IN r.sources AND 'Haydock' IN r.sources THEN 'Both'
            WHEN 'TSK' IN r.sources THEN 'TSK-only'
            WHEN 'Haydock' IN r.sources THEN 'Haydock-only'
            ELSE 'Unknown'
          END AS source_type,
          count(r) AS count
        ORDER BY count DESC
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    
    print("Deuterocanonical Edge Sources:")
    print("=" * 35)
    for record in result:
        print(f"  {record['source_type']:15} {record['count']:>10,}")
    print("\n  Note: TSK only appears here when DC book connects to a")
    print("  Protestant-canon verse that TSK also references.")

Deuterocanonical Edge Sources:
  Haydock-only           857

  Note: TSK only appears here when DC book connects to a
  Protestant-canon verse that TSK also references.


### Sample Deuterocanonical Cross-References

Let's look at some specific connections from the deuterocanonical books to see what kinds of references they provide.

In [15]:
with conn.session() as session:
    # Get sample refs FROM deuterocanonical books TO non-DC books
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE a.book_id IN dc_books AND NOT b.book_id IN dc_books
        RETURN a.id AS from_id, a.book_name AS from_book, a.text AS from_text,
               b.id AS to_id, b.book_name AS to_book, b.text AS to_text
        LIMIT 10
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    
    print("Sample Cross-References FROM Deuterocanonical Books:")
    print("=" * 80)
    for record in result:
        print(f"\n{record['from_id']} ({record['from_book']})")
        print(f"  -> {record['to_id']} ({record['to_book']})")
        print(f"  FROM: {record['from_text'][:65]}...")
        print(f"  TO:   {record['to_text'][:65]}...")

Sample Cross-References FROM Deuterocanonical Books:

SIR-1-1 (Sirach)
  -> 1KI-3-9 (I Kings)
  FROM: All wisdom is from the Lord God, and has always been with him, an...
  TO:   Therefore, give to your servant a teachable heart, so that he may...

SIR-1-1 (Sirach)
  -> 1KI-4-29 (I Kings)
  FROM: All wisdom is from the Lord God, and has always been with him, an...
  TO:   And God gave wisdom to Solomon, and an exceedingly great prudence...

SIR-1-16 (Sirach)
  -> PSA-110-10 (Psalms)
  FROM: The fear of the Lord is the beginning of wisdom, and was created ...
  TO:   The fear of the Lord is the beginning of wisdom. A good understan...

SIR-1-16 (Sirach)
  -> PRO-1-7 (Proverbs)
  FROM: The fear of the Lord is the beginning of wisdom, and was created ...
  TO:   The fear of the Lord is the beginning of wisdom. The foolish desp...

SIR-1-16 (Sirach)
  -> PRO-9-10 (Proverbs)
  FROM: The fear of the Lord is the beginning of wisdom, and was created ...
  TO:   The fear of the Lord is the begi

## Visualizations

### Network Around a Deuterocanonical Verse

Let's visualize the cross-reference network around a verse from Sirach (Ecclesiasticus), one of the most connected deuterocanonical books.

In [16]:
def visualize_verse_network(verse_id, limit=40):
    """Create an interactive visualization of a verse and its connections.
    
    Deuterocanonical books are highlighted in teal.
    """
    net = Network(
        height="600px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white",
        notebook=True,
        cdn_resources='in_line'
    )
    net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=200)
    
    query = """
    MATCH (center:Verse {id: $verse_id})
    OPTIONAL MATCH (center)-[r:CROSS_REFERENCES]-(connected:Verse)
    WITH center, connected, r
    LIMIT $limit
    RETURN center.id AS center_id, center.book_id AS center_book, 
           center.text AS center_text,
           connected.id AS conn_id, connected.book_id AS conn_book,
           connected.text AS conn_text,
           startNode(r).id AS from_id, endNode(r).id AS to_id,
           r.sources AS sources
    """
    
    added_nodes = set()
    
    with conn.session() as session:
        results = session.run(query, verse_id=verse_id, limit=limit)
        
        for record in results:
            # Add center node
            center_id = record["center_id"]
            if center_id not in added_nodes:
                is_dc = record["center_book"] in DEUTEROCANONICAL_BOOKS
                net.add_node(
                    center_id,
                    label=center_id,
                    title=record["center_text"][:200],
                    color=get_book_color(record["center_book"]),
                    size=35 if is_dc else 30,
                    borderWidth=3 if is_dc else 1,
                )
                added_nodes.add(center_id)
            
            # Add connected node
            conn_id = record["conn_id"]
            if conn_id and conn_id not in added_nodes:
                is_dc = record["conn_book"] in DEUTEROCANONICAL_BOOKS
                net.add_node(
                    conn_id,
                    label=conn_id,
                    title=record["conn_text"][:200] if record["conn_text"] else "",
                    color=get_book_color(record["conn_book"]),
                    size=25 if is_dc else 20,
                    borderWidth=3 if is_dc else 1,
                )
                added_nodes.add(conn_id)
            
            # Add edge with source info
            if record["from_id"] and record["to_id"]:
                sources = record["sources"] or []
                source_str = ", ".join(sources)
                # Color edge by source
                if "TSK" in sources and "Haydock" in sources:
                    edge_color = "#2ecc71"  # Green for both
                elif "Haydock" in sources:
                    edge_color = "#16a085"  # Teal for Haydock
                else:
                    edge_color = "#3498db"  # Blue for TSK
                
                net.add_edge(
                    record["from_id"],
                    record["to_id"],
                    title=f"Source: {source_str}",
                    color=edge_color,
                    width=2,
                )
    
    print(f"Graph: {len(added_nodes)} nodes")
    print("Edge colors: Blue=TSK, Teal=Haydock, Green=Both")
    return net

In [17]:
# Visualize Sirach 6:14 - "A faithful friend is a strong defense"
net = visualize_verse_network("SIR-6-14", limit=30)
net.show("sirach_6_14_network.html")

Graph: 1 nodes
Edge colors: Blue=TSK, Teal=Haydock, Green=Both
sirach_6_14_network.html


In [18]:
# Visualize Wisdom 2:24 - "By the envy of the devil, death entered the world"
net = visualize_verse_network("WIS-2-24", limit=30)
net.show("wisdom_2_24_network.html")

Graph: 2 nodes
Edge colors: Blue=TSK, Teal=Haydock, Green=Both
wisdom_2_24_network.html


### Book-Level Connections Highlighting Deuterocanonicals

Visualize connections between all 73 books, with deuterocanonical books highlighted.

In [19]:
def visualize_book_connections(min_connections=200, include_dc_emphasis=True):
    """Create a graph showing connections between books.
    
    Deuterocanonical books are larger and have a distinct border.
    """
    net = Network(
        height="700px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white",
        notebook=True,
        cdn_resources='in_line'
    )
    net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=300)
    
    query = """
    MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
    WHERE a.book_id <> b.book_id
    WITH a.book_id AS from_book, b.book_id AS to_book, count(r) AS connections
    WHERE connections >= $min_connections
    RETURN from_book, to_book, connections
    ORDER BY connections DESC
    """
    
    added_nodes = set()
    
    with conn.session() as session:
        results = session.run(query, min_connections=min_connections)
        
        for record in results:
            from_book = record["from_book"]
            to_book = record["to_book"]
            connections = record["connections"]
            
            # Add book nodes
            for book in [from_book, to_book]:
                if book not in added_nodes:
                    is_dc = book in DEUTEROCANONICAL_BOOKS
                    net.add_node(
                        book,
                        label=book,
                        title=DC_BOOK_NAMES.get(book, book) if is_dc else book,
                        color=get_book_color(book),
                        size=35 if is_dc else 25,
                        borderWidth=4 if is_dc else 1,
                        borderWidthSelected=6 if is_dc else 2,
                    )
                    added_nodes.add(book)
            
            # Add edge
            # Highlight edges involving DC books
            involves_dc = from_book in DEUTEROCANONICAL_BOOKS or to_book in DEUTEROCANONICAL_BOOKS
            edge_color = "#16a085" if involves_dc else "#666666"
            
            net.add_edge(
                from_book,
                to_book,
                title=f"{connections:,} refs",
                width=max(1, connections / 300),
                color=edge_color,
            )
    
    print(f"Graph: {len(added_nodes)} books")
    print("Teal edges involve deuterocanonical books")
    return net

In [20]:
net = visualize_book_connections(min_connections=100)
net.show("book_connections_with_dc.html")

Graph: 60 books
Teal edges involve deuterocanonical books
book_connections_with_dc.html


### TSK vs Haydock Book Connections Comparison

Create separate visualizations for TSK-only and Haydock-only book connections.

In [21]:
def visualize_source_book_connections(source, min_connections=100):
    """Create a book connection graph for a specific source."""
    net = Network(
        height="600px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white",
        notebook=True,
        cdn_resources='in_line'
    )
    net.barnes_hut(gravity=-5000, central_gravity=0.4, spring_length=250)
    
    # Query for source-specific connections
    query = """
    MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
    WHERE a.book_id <> b.book_id AND $source IN r.sources
    WITH a.book_id AS from_book, b.book_id AS to_book, count(r) AS connections
    WHERE connections >= $min_connections
    RETURN from_book, to_book, connections
    ORDER BY connections DESC
    """
    
    added_nodes = set()
    edge_color = "#16a085" if source == "Haydock" else "#3498db"
    
    with conn.session() as session:
        results = session.run(query, source=source, min_connections=min_connections)
        
        for record in results:
            from_book = record["from_book"]
            to_book = record["to_book"]
            connections = record["connections"]
            
            for book in [from_book, to_book]:
                if book not in added_nodes:
                    is_dc = book in DEUTEROCANONICAL_BOOKS
                    net.add_node(
                        book,
                        label=book,
                        color=get_book_color(book),
                        size=30 if is_dc else 22,
                        borderWidth=3 if is_dc else 1,
                    )
                    added_nodes.add(book)
            
            net.add_edge(
                from_book,
                to_book,
                title=f"{connections:,} refs",
                width=max(1, connections / 200),
                color=edge_color,
            )
    
    print(f"{source} connections: {len(added_nodes)} books")
    return net

In [22]:
# TSK book connections
net_tsk = visualize_source_book_connections("TSK", min_connections=500)
net_tsk.show("book_connections_tsk.html")

TSK connections: 38 books
book_connections_tsk.html


In [23]:
# Haydock book connections (lower threshold since fewer total refs)
net_haydock = visualize_source_book_connections("Haydock", min_connections=20)
net_haydock.show("book_connections_haydock.html")

Haydock connections: 23 books
book_connections_haydock.html


## Summary

Key findings from this analysis:

1. **Source Distribution**: TSK provides the majority of cross-references, but Haydock adds unique connections especially for the deuterocanonical books.

2. **Canon Impact**: Including the 7 deuterocanonical books adds significant cross-references that connect to the broader scriptural narrative.

3. **Different Emphases**: TSK and Haydock often emphasize different books and verses, reflecting their different methodologies (English word-matching vs. patristic/theological tradition).

4. **High-Confidence Connections**: Edges attested by both TSK and Haydock represent particularly strong connections worth exploring.

## Cleanup

In [24]:
conn.close()
print("Connection closed")

Connection closed
